# 02 - Stock Predictor Engine

Goal: forecast forward returns and price direction using leakage-safe features and time-based validation.

In [6]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))

import joblib
import pandas as pd

from invest_intel.data import load_processed
from invest_intel.features import drop_modeling_na, modeling_feature_columns
from invest_intel.modeling import train_direction_model, train_return_models

In [7]:
features = load_processed('nifty50_features.csv')
target = 'future_return_5d'
feature_cols = modeling_feature_columns(features)
model_df = drop_modeling_na(features, target, feature_cols)
model_df[['date', 'symbol', target]].head(), len(feature_cols), model_df.shape

(        date      symbol  future_return_5d
 0 2012-06-08  ADANIPORTS         -0.080529
 1 2012-06-11  ADANIPORTS         -0.068338
 2 2012-06-12  ADANIPORTS         -0.062016
 3 2012-06-13  ADANIPORTS         -0.027732
 4 2012-06-14  ADANIPORTS          0.026638,
 40,
 (119475, 52))

## Return Forecasting

In [8]:
return_results = train_return_models(model_df, feature_cols, target_col=target, test_start_date='2019-01-01')
pd.DataFrame([{'model': result.name, **result.metrics} for result in return_results])

,model,mae,rmse,r2,directional_accuracy
0,ridge_regression,0.039347,0.058356,-0.027511,0.495897
1,hist_gradient_boosting,0.041109,0.060506,-0.104596,0.499001


## Direction Classification

In [9]:
direction_df = drop_modeling_na(features, 'target_direction_5d', feature_cols)
direction_result = train_direction_model(direction_df, feature_cols, target_col='target_direction_5d')
direction_result.metrics

{'directional_accuracy': 0.5061367204224347}

In [10]:
best = max(return_results, key=lambda result: result.metrics['directional_accuracy'])
(ROOT / 'models').mkdir(exist_ok=True)
joblib.dump({'model': best.estimator, 'feature_cols': feature_cols, 'target': target}, ROOT / 'models' / 'stock_return_model.joblib')
joblib.dump({'model': direction_result.estimator, 'feature_cols': feature_cols, 'target': 'target_direction_5d'}, ROOT / 'models' / 'stock_direction_model.joblib')

['e:\\Projects\\cult_invest_intel_ml\\models\\stock_direction_model.joblib']